The dataset being used for this Project can be found here: https://www.kaggle.com/datasets/vijayuv/onlineretail/data
From the given data set of online retail dataset, I would like to understand the customer behavior, product performance and revenue trends to get actionable insights to increase profitabilty and improve customer retention.
The main objectives for this projects would be:
1. Understand sales patterns
2. Identify top-performing products
3. Analyze customer purchasing behavior
4. Discover revenue trends
5. Segment customers using RFM Analysis
6. Generate actionable business recommendations

In [1]:
#Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Loading Online Retail Dataset and returning first three rows

df_raw = pd.read_csv("OnlineRetail.csv", encoding="latin1")
df_raw.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom


Exploratory Data Analysis:

In [3]:
# Getting basic information about dataset

df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


Looking at the type of columns, we can see that InvoiceDate Column is wrong, we will change it to datetime format

In [4]:
#Invoice Date type changed from str to datetime

df_raw['InvoiceDate'] = pd.to_datetime(df_raw['InvoiceDate'])
df_raw['InvoiceDate'].dtype


dtype('<M8[us]')

In [5]:
df_raw.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


From the describe output, Quantity and UnitPrice column has negative values, We will analyse this further
Lets start with Quantity first abd figure out what percentage of data is negative

In [6]:
print(f"The number of negative quantity for quantity is {(df_raw['Quantity']<0).sum()} and it is {((df_raw['Quantity']<0).sum()/len(df_raw))*100:.2f}%/")

The number of negative quantity for quantity is 10624 and it is 1.96%/


In [7]:
print(f"The number of negative quantity for Unit Price is {(df_raw['UnitPrice']<0).sum()} and it is {((df_raw['UnitPrice']<0).sum()/len(df_raw))*100:.2f}%/")


The number of negative quantity for Unit Price is 2 and it is 0.00%/


Now lets have a quick look to see if there are any Quantity and Unit Price which are zero values

In [8]:
print(f"Total number of records with zero quantity is {(df_raw['Quantity'] == 0).sum()} with {((df_raw['Quantity']==0).sum()/len(df_raw))*100:.2f}%" )
print(f"Total number of records with zero unit price is {(df_raw['UnitPrice'] == 0).sum()} with {((df_raw['UnitPrice']==0).sum()/len(df_raw))*100:.2f}%")

Total number of records with zero quantity is 0 with 0.00%
Total number of records with zero unit price is 2515 with 0.46%


We have done the EDA phase and following are the findings from the EDA:
1. One date column has been changed to datetime format
2. Negative values are present for Quantity and Unit Price which account for 2.5 % of total data
3. Zero values are present for Unit price which account for 0.5% of total data

Next we will remove duplicate data and check how to handle Null Values

In [9]:
#Remove duplicate values:
df_raw.drop_duplicates(inplace=True)
df_raw.info()

<class 'pandas.DataFrame'>
Index: 536641 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    536641 non-null  str           
 1   StockCode    536641 non-null  str           
 2   Description  535187 non-null  str           
 3   Quantity     536641 non-null  int64         
 4   InvoiceDate  536641 non-null  datetime64[us]
 5   UnitPrice    536641 non-null  float64       
 6   CustomerID   401604 non-null  float64       
 7   Country      536641 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 36.8 MB


In [10]:
df_raw.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
dtype: int64

Only Description and Customer ID has null values, description Null values we can ignore, CustomerID null values we will have to handle based on our analysis need

In [11]:
# Data Quality Summary

total_records = len(df_raw)
missing_customerid = df_raw['CustomerID'].isnull().sum()
negative_quantity = (df_raw['Quantity'] < 0).sum()
negative_unit_price = (df_raw['UnitPrice'] < 0).sum()
zero_unit_price = (df_raw['UnitPrice'] == 0).sum()
# Create summary table
data_quality_summary = pd.DataFrame({'Data Quality Metric': ['Total Records','Missing CustomerID','Negative Quantity','Negative Unit Price','Zero Unit Price'],
    'Count': [total_records, missing_customerid, negative_quantity, negative_unit_price, zero_unit_price]})

# Calculate percentage of total dataset
data_quality_summary['% of Dataset'] = (
    data_quality_summary['Count'] / total_records * 100
).round(2)

# Format percentage column
data_quality_summary['% of Dataset'] = (
    data_quality_summary['% of Dataset'].astype(str) + '%'
)

# Display the summary
data_quality_summary


,Data Quality Metric,Count,% of Dataset
0,Total Records,536641,100.0%
1,Missing CustomerID,135037,25.16%
2,Negative Quantity,10587,1.97%
3,Negative Unit Price,2,0.0%
4,Zero Unit Price,2510,0.47%


Data Cleanup:

Negative Quantity, Negative Unit Price and Zero Unit Price:
we can drop the above items for our analysis for the following reasons:
1. Transactions with negative quantities typically correspond to returns or credit notes. These records do not represent new purchases and can reduce revenue, inflate purchase frequency if counted as separate invoices, and misrepresent customer value. Therefore, they were excluded to ensure the analysis reflects actual purchasing behavior.

2. Transactions with a zero unit price typically correspond to free items, promotional giveaways, or system-generated records. These transactions do not represent revenue-generating sales and can distort sales metrics and customer value analysis. Therefore, they were excluded to ensure the analysis reflects actual purchasing behavior.

Null CustomerID records:

Approximately 25% of the transactions have missing CustomerID values. These records were retained for sales and product analysis because they represent valid transactions contributing to overall business performance. However, they were excluded from the RFM analysis, as a unique customer identifier is required to calculate Recency, Frequency, and Monetary metrics and assign customers to meaningful segments.
So we will two cleaned data frame:

1. df_clean_sales: We will drop Negative quantity records and unit price along with zero unit price items, to be used for Sales, Revevuew and Product Analysis
2. df_clean_RFM: We will remove the null consumer id along with negative quantity and unit price and zero unit price items

In [12]:
df_clean_sales = df_raw[(df_raw['Quantity'] > 0) & (df_raw['UnitPrice'] >= 0)].copy()
df_clean_sales.info()

<class 'pandas.DataFrame'>
Index: 526052 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    526052 non-null  str           
 1   StockCode    526052 non-null  str           
 2   Description  525460 non-null  str           
 3   Quantity     526052 non-null  int64         
 4   InvoiceDate  526052 non-null  datetime64[us]
 5   UnitPrice    526052 non-null  float64       
 6   CustomerID   392732 non-null  float64       
 7   Country      526052 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 36.1 MB


In [13]:
df_clean_RFM = df_clean_sales.dropna(subset=['CustomerID']).copy()
df_clean_RFM.info()

<class 'pandas.DataFrame'>
Index: 392732 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392732 non-null  str           
 1   StockCode    392732 non-null  str           
 2   Description  392732 non-null  str           
 3   Quantity     392732 non-null  int64         
 4   InvoiceDate  392732 non-null  datetime64[us]
 5   UnitPrice    392732 non-null  float64       
 6   CustomerID   392732 non-null  float64       
 7   Country      392732 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 27.0 MB
